In [8]:
import yfinance as yf

In [ ]:
import pandas as pd

# Top 10 US stocks by market cap (typical order; use yfinance for live P/E)
# Sources: major indices / market cap rankings (NVDA, AAPL, MSFT, GOOGL, AMZN, META, AVGO, TSLA, BRK-B, LLY)
tickers = ["NVDA", "AAPL", "MSFT", "GOOGL", "AMZN", "META", "AVGO", "TSLA", "BRK-B", "LLY"]
print("Top 10 US stocks by market cap (tickers):", tickers)

In [9]:
# Top 10 by P/E ratio (ascending – lowest P/E first), using financials.ratios.pe
from financials.ratios import pe
import yfinance as yf

pe_data = []
for ticker in tickers:
    t = yf.Ticker(ticker)
    info = t.info
    name = info.get("shortName", ticker)
    pe_data.append({
        "Ticker": ticker,
        "Company": name,
        "P/E": pe(ticker),
    })

df_pe = pd.DataFrame(pe_data)
df_pe = df_pe.sort_values(by="P/E", ascending=True, na_position="last")
df_pe

NameError: name 'tickers' is not defined

In [6]:
# Valuation ratios: P/E, PEG (5yr expected), Price/Sales, EV/EBITDA
from financials.ratios import pe, peg_ratio, price_to_sales, ev_to_ebitda
import yfinance as yf

ratio_data = []
for ticker in tickers:
    t = yf.Ticker(ticker)
    info = t.info
    ratio_data.append({
        "Ticker": ticker,
        "Company": info.get("shortName", ticker),
        "P/E": pe(ticker),
        "PEG Ratio (5yr expected)": peg_ratio(ticker),
        "Price/Sales": price_to_sales(ticker),
        "EV/EBITDA": ev_to_ebitda(ticker),
    })

df_ratios = pd.DataFrame(ratio_data)

# 4 views: P/E ascending, PEG descending, Price/Sales ascending, EV/EBITDA descending
print("1. By P/E (ascending – lowest first)")
display(df_ratios.sort_values(by="P/E", ascending=True, na_position="last")[["Ticker", "Company", "P/E"]])

print("2. By PEG Ratio (5yr expected) (descending)")
display(df_ratios.sort_values(by="PEG Ratio (5yr expected)", ascending=False, na_position="last")[["Ticker", "Company", "PEG Ratio (5yr expected)"]])

print("3. By Price/Sales (ascending)")
display(df_ratios.sort_values(by="Price/Sales", ascending=True, na_position="last")[["Ticker", "Company", "Price/Sales"]])

print("4. By EV/EBITDA (descending)")
display(df_ratios.sort_values(by="EV/EBITDA", ascending=False, na_position="last")[["Ticker", "Company", "EV/EBITDA"]])

NameError: name 'tickers' is not defined

In [ ]:
# Diluted EPS for top 10 stocks – last 5 years (using financials.eps)
from financials.eps import annual_eps

years = list(range(pd.Timestamp.now().year - 4, pd.Timestamp.now().year))
rows = []
for ticker in tickers:
    row = {"Ticker": ticker}
    for y in years:
        row[y] = annual_eps(ticker, y)
    rows.append(row)

df_diluted_eps = pd.DataFrame(rows).set_index("Ticker")
# Add change 2024 → 2025 and sort by it (descending)
if 2024 in df_diluted_eps.columns and 2025 in df_diluted_eps.columns:
    df_diluted_eps["Change (2024→2025)"] = df_diluted_eps[2025] - df_diluted_eps[2024]
    df_diluted_eps = df_diluted_eps.sort_values(by="Change (2024→2025)", ascending=False, na_position="last")
else:
    last_year = years[-1]
    df_diluted_eps = df_diluted_eps.sort_values(by=last_year, ascending=False, na_position="last")
df_diluted_eps

In [ ]:
# Quarterly diluted EPS for top 10 stocks – last 3 years (each quarter)
import financials.eps
import importlib
importlib.reload(financials.eps)
from financials.eps import quarterly_eps

years = list(range(pd.Timestamp.now().year - 2, pd.Timestamp.now().year))
quarters = ["Q1", "Q2", "Q3", "Q4"]
cols = [f"{y} {q}" for y in years for q in quarters]

rows = []
for ticker in tickers:
    row = {"Ticker": ticker}
    for y in years:
        for q in quarters:
            row[f"{y} {q}"] = quarterly_eps(ticker, y, q)
    rows.append(row)

df_quarterly_eps = pd.DataFrame(rows).set_index("Ticker")
# Same descending order by last year EPS as annual table
df_quarterly_eps = df_quarterly_eps.reindex(df_diluted_eps.index)
df_quarterly_eps

In [14]:
# Example: use insider package to inspect NVDA insider activity
from insider import (
    insider_trades_df,
    collapse_form4_trades,
    filter_open_market,
    net_insider_flow,
)

# Fetch raw Form 4 data
trades_df = insider_trades_df("ANET", limit=50)

# Optional: cleaned, one row per logical trade
collapsed = collapse_form4_trades(trades_df)
print("\nCollapsed trades (head):")
print(collapsed.head())

# Optional: open-market only
open_market = filter_open_market(collapsed)
print("\nOpen‑market only (head):")
print(open_market.head())

# Optional: 90d net flow
print("\nNet insider flow (90d):", net_insider_flow(trades_df, days=90))



Collapsed trades (head):
   ticker                      insider                       title  \
38   ANET                 Duda Kenneth           President and CTO   
39   ANET                 Duda Kenneth           President and CTO   
40   ANET                 Duda Kenneth           President and CTO   
7    ANET  Breithaupt Chantelle Yvette  Senior Vice President, CFO   
49   ANET          Giancarlo Charles H                         NaN   

   filing_date transaction_date transaction_code direct_or_indirect   shares  \
38  2026-02-19       2026-02-17                M                  D  64000.0   
39  2026-02-19       2026-02-17                S                  D  32000.0   
40  2026-02-19       2026-02-17                S                  I  26000.0   
7   2026-02-17       2026-02-13                A                  D  19170.0   
49  2026-02-04       2026-02-02                S                  I   8000.0   

           value  shares_owned_after  price_vwap  
38  4.884000e+05     

In [1]:
# Example: rank by Price/Sales (ascending) from nasdaq100 CSV
from financials.ratios import pe
from screening import rank_stocks_by_ratio

df = rank_stocks_by_ratio(pe, csv_path="sp500_symbols.csv", ascending=True)
print(df.head(20))

   ticker     ratio
0    CHTR  3.615801
1   CMCSA  4.543138
2     ALL  5.302280
3     UHS  6.083090
4     EXE  6.582590
5      EG  7.240733
6     FIS  7.474806
7    ACGL  7.503077
8     AES  7.640625
9       T  7.649831
10    APA  7.694638
11    SYF  8.142858
12    EIX  8.225000
13   PYPL  8.309568
14    HPQ  8.474073
15    MKC  8.494176
16   CTSH  8.683296
17    GIS  8.804400
18   LDOS  9.310156
19   VICI  9.318493


In [2]:
# Example: CANSLIM ranking for nasdaq symbols
from screening import rank_stocks_by_canslim

df = rank_stocks_by_canslim(
    csv_path="sp500_symbols.csv",
    limit=5000,
    weights={"C": 20, "A": 15, "N": 15, "S": 10, "L": 25, "I": 10, "M": 5},
)
print(df.head(25))

   ticker           C           A          N           S           L  \
0     TSN  100.000000   93.487268  78.679606    5.071170   16.319617   
1     AFL   94.599327  100.000000  99.715742    4.964147   22.729760   
2    DELL    9.548822    9.622575  91.638262    6.535617  100.000000   
3      MU   40.309764   37.057902  67.202946   14.987004   62.201388   
4     FIS   82.717845   79.299717  38.656358   12.598047    8.093143   
5     HPE   50.000000    2.645387  68.243153    5.188586   55.066674   
6     VLO   50.000000    2.886267  93.205227   13.344978   37.280925   
7     MPC   50.000000    3.179517  93.367504   13.455700   41.466789   
8    PANW   50.000000    2.607897  84.983295   11.012276   41.292607   
9     PNW   50.000000    2.727339  94.379163   10.995270   22.789198   
10   VTRS   50.000000    2.436414  96.968905   11.501049   27.642793   
11    SJM   50.000000    2.425576  98.933411   10.207412   22.652446   
12    CZR   50.000000    2.215137  96.283806    0.000000   25.87

In [1]:
from screening import top_decile_momentum

# Default: top 10% by 0.7*RS6 + 0.3*RS12
top = top_decile_momentum("sp500", top_pct=0.02)
print(top[["ticker", "momentum_score", "rank"]])

top = top_decile_momentum("nasdaq100")
print(top[["ticker", "momentum_score", "rank"]])

  ticker  momentum_score  rank
0     MU        7.246622     1
1    WDC        5.794851     2
2    ITW        5.269312     3
3    STX        4.964096     4
4   DELL        4.044143     5
5   INTC        3.668520     6
6    AMD        2.634673     7
7    TER        2.226529     8
8     ON        2.210174     9
9    GLW        2.019921    10
  ticker  momentum_score  rank
0    WDC        3.218163     1
1     MU        3.164046     2
2   INTC        2.507075     3
3   MRVL        1.934076     4
4    ARM        1.582901     5
5    AMD        1.323505     6
6   LRCX        1.155578     7
7   AMAT        1.138504     8
8   KLAC        0.771166     9
